In [ ]:
!pip install --upgrade pip
!pip install ultralytics roboflow pandas matplotlib pyyaml

In [ ]:
from roboflow import Roboflow
import os, shutil

API_KEY = "lCakLx5MAemDQu7MOe0K"
WORKSPACE = "arjuns-workspace-utjh6"
PROJECT = "data-science-3-tlvei"
VERSION = 1

LOCAL_DATASET_ROOT = r"C:\Users\Arjun\Documents\yolo_vehicle_project\dataset_raw"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
version = project.version(VERSION)

try:
    dataset = version.download("yolo26")
except: 
    print("Error occurred while downloading the dataset.")
    dataset = None

if os.path.abspath(dataset.location) != os.path.abspath(LOCAL_DATASET_ROOT):
    if os.path.exists(LOCAL_DATASET_ROOT):
        shutil.rmtree(LOCAL_DATASET_ROOT)
    shutil.copytree(dataset.location, LOCAL_DATASET_ROOT)

print("Final dataset root:", LOCAL_DATASET_ROOT)

In [ ]:
import os, glob, random, shutil
random.seed(42)

root = r"C:\Users\Arjun\Documents\yolo_vehicle_project\dataset_raw"
train_img = os.path.join(root, "train", "images")
train_lbl = os.path.join(root, "train", "labels")
valid_img = os.path.join(root, "valid", "images")
valid_lbl = os.path.join(root, "valid", "labels")

os.makedirs(valid_img, exist_ok=True)
os.makedirs(valid_lbl, exist_ok=True)

# current counts
train_existing = glob.glob(os.path.join(train_img, "*"))
valid_existing = glob.glob(os.path.join(valid_img, "*"))
print("Before split -> train:", len(train_existing), "valid:", len(valid_existing))

if len(valid_existing) == 0:
    imgs = []
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"):
        imgs.extend(glob.glob(os.path.join(train_img, ext)))
    imgs = sorted(imgs)

    if len(imgs) == 0:
        raise RuntimeError("No images found in train/images. Check dataset path.")

    n_valid = max(1, int(len(imgs) * 0.2))
    n_valid = min(n_valid, len(imgs))
    selected = set(random.sample(imgs, n_valid))

    moved = 0
    missing_labels = 0

    for ip in selected:
        name = os.path.basename(ip)
        stem = os.path.splitext(name)[0]
        lp = os.path.join(train_lbl, f"{stem}.txt")

        shutil.move(ip, os.path.join(valid_img, name))
        moved += 1

        if os.path.exists(lp):
            shutil.move(lp, os.path.join(valid_lbl, f"{stem}.txt"))
        else:
            missing_labels += 1

    print(f"Moved {moved} images to valid.")
    print(f"Missing labels among moved images: {missing_labels}")
else:
    print("Valid split already exists. Skipping split.")

print("After split -> train:", len(glob.glob(os.path.join(train_img, '*'))))
print("After split -> valid:", len(glob.glob(os.path.join(valid_img, '*'))))

In [ ]:
import os, yaml

root = r"C:\Users\Arjun\Documents\yolo_vehicle_project\dataset_raw"
yaml_path = os.path.join(root, "data_fixed.yaml")

data = {
    "path": root.replace("\\", "/"),
    "train": "train/images",
    "val": "valid/images",
    "names": {
        0: "jeepneys",
        1: "bus",
        2: "car",
        3: "motorcycle",
        4: "trucks",
        5: "vans"
    }
}

with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data, f, sort_keys=False, allow_unicode=True)

print("Saved:", yaml_path)
with open(yaml_path, "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
from ultralytics import YOLO
import os

yaml_path = r"C:\Users\Arjun\Documents\yolo_vehicle_project\dataset_raw\data_fixed.yaml"
PROJECT_DIR = r"C:\Users\Arjun\Documents\yolo_vehicle_project\outputs"
os.makedirs(PROJECT_DIR, exist_ok=True)

experiments = [
    {"name":"model_A", "model":"yolo26n.pt", "epochs":25, "imgsz":640, "optimizer":"AdamW", "batch":4,  "lr0":0.01},
    {"name":"model_B", "model":"yolo26n.pt", "epochs":30, "imgsz":640, "optimizer":"SGD",   "batch":20, "lr0":0.001},
    {"name":"model_C", "model":"yolo26n.pt", "epochs":40, "imgsz":640, "optimizer":"auto",  "batch":-1, "lr0":0.0001},
]

for exp in experiments:
    model = YOLO(exp["model"])
    model.train(
        data=yaml_path,
        epochs=exp["epochs"],
        imgsz=exp["imgsz"],
        optimizer=exp["optimizer"],
        batch=exp["batch"],
        lr0=exp["lr0"],
        project=PROJECT_DIR,
        name=exp["name"],
        pretrained=True,
        verbose=True
    )

In [ ]:
import os, pandas as pd
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

PROJECT_DIR = r"C:\Users\Arjun\Documents\yolo_vehicle_project\outputs"
yaml_path = r"C:\Users\Arjun\Documents\yolo_vehicle_project\dataset_raw\data_fixed.yaml"

experiments = [
    {"name":"model_A", "model":"yolo26n.pt", "epochs":25, "imgsz":640, "optimizer":"AdamW", "batch":4,  "lr0":0.01},
    {"name":"model_B", "model":"yolo26n.pt", "epochs":30, "imgsz":640, "optimizer":"SGD",   "batch":20, "lr0":0.001},
    {"name":"model_C", "model":"yolo26n.pt", "epochs":40, "imgsz":640, "optimizer":"auto",  "batch":-1, "lr0":0.0001},
]

final_rows = []
for exp in experiments:
    csv_path = os.path.join(PROJECT_DIR, exp["name"], "results.csv")
    df = pd.read_csv(csv_path)
    row = df.iloc[-1].to_dict()
    row["Model Run"] = exp["name"]
    final_rows.append(row)

final_epoch_df = pd.DataFrame(final_rows)
final_epoch_df.to_csv(os.path.join(PROJECT_DIR, "deliverable_final_epoch_results.csv"), index=False)

hyper_df = pd.DataFrame(experiments)[["name","model","epochs","imgsz","optimizer","batch","lr0"]]
hyper_df = hyper_df.rename(columns={"name":"Model Run", "lr0":"Learning Rate"})
hyper_df.to_csv(os.path.join(PROJECT_DIR, "deliverable_hyperparameters_table.csv"), index=False)

metrics_rows = []
for exp in experiments:
    best_w = os.path.join(PROJECT_DIR, exp["name"], "weights", "best.pt")
    m = YOLO(best_w).val(data=yaml_path, imgsz=640, verbose=False)

    p = float(m.box.mp)
    r = float(m.box.mr)
    map50 = float(m.box.map50)
    f1 = 2 * p * r / (p + r + 1e-9)

    metrics_rows.append({
        "Model Run": exp["name"],
        "mAP50": map50,
        "Precision": p,
        "Recall": r,
        "F1 Score": f1
    })

metrics_df = pd.DataFrame(metrics_rows).sort_values("mAP50", ascending=False)
metrics_df.to_csv(os.path.join(PROJECT_DIR, "deliverable_metrics_table.csv"), index=False)

print("Saved deliverables in:", PROJECT_DIR)
print(final_epoch_df)
print(hyper_df)
print(metrics_df)

for exp in experiments:
    cm_path = os.path.join(PROJECT_DIR, exp["name"], "confusion_matrix.png")
    if os.path.exists(cm_path):
        img = mpimg.imread(cm_path)
        plt.figure(figsize=(7,7))
        plt.title(f"{exp['name']} Confusion Matrix")
        plt.imshow(img)
        plt.axis("off")
        plt.show()

In [ ]:
import pandas as pd

hyperparams = pd.DataFrame([
    {"Model": "Model A", "Model File": "yolo26n.pt", "Epochs": 25, "Image Size": 640, "Optimizer": "AdamW", "Batch Size": 4,  "Initial LR (lr0)": 0.01},
    {"Model": "Model B", "Model File": "yolo26n.pt", "Epochs": 30, "Image Size": 640, "Optimizer": "SGD",   "Batch Size": 20, "Initial LR (lr0)": 0.001},
    {"Model": "Model C", "Model File": "yolo26n.pt", "Epochs": 40, "Image Size": 640, "Optimizer": "auto",  "Batch Size": -1, "Initial LR (lr0)": 0.0001},
])
hyperparams

In [ ]:
overall_metrics = pd.DataFrame([
    {"Model": "Model A", "Precision": 0.751, "Recall": 0.669, "mAP50": 0.727, "mAP50-95": 0.497},
    {"Model": "Model B", "Precision": 0.540, "Recall": 0.453, "mAP50": 0.461, "mAP50-95": 0.310},
    {"Model": "Model C", "Precision": 0.767, "Recall": 0.680, "mAP50": 0.745, "mAP50-95": 0.526},
])
overall_metrics["F1 Score"] = (2 * overall_metrics["Precision"] * overall_metrics["Recall"]) / (overall_metrics["Precision"] + overall_metrics["Recall"])
overall_metrics

In [ ]:
per_class = pd.DataFrame([

    {"Model":"Model A","Class":"jeepneys","Instances":480,"Precision":0.552,"Recall":0.588,"mAP50":0.545},
    {"Model":"Model A","Class":"bus","Instances":279,"Precision":0.500,"Recall":0.641,"mAP50":0.564},
    {"Model":"Model A","Class":"car","Instances":8965,"Precision":0.667,"Recall":0.527,"mAP50":0.608},
    {"Model":"Model A","Class":"motorcycle","Instances":4048,"Precision":0.420,"Recall":0.495,"mAP50":0.450},
    {"Model":"Model A","Class":"trucks","Instances":580,"Precision":0.691,"Recall":0.598,"mAP50":0.650},
    {"Model":"Model A","Class":"vans","Instances":1377,"Precision":0.532,"Recall":0.434,"mAP50":0.438},

    {"Model":"Model B","Class":"jeepneys","Instances":480,"Precision":0.000,"Recall":0.000,"mAP50":0.000},
    {"Model":"Model B","Class":"bus","Instances":279,"Precision":0.251,"Recall":0.099,"mAP50":0.087},
    {"Model":"Model B","Class":"car","Instances":8965,"Precision":1.000,"Recall":0.000,"mAP50":0.002},
    {"Model":"Model B","Class":"motorcycle","Instances":4048,"Precision":1.000,"Recall":0.000,"mAP50":0.003},
    {"Model":"Model B","Class":"trucks","Instances":580,"Precision":0.255,"Recall":0.548,"mAP50":0.333},
    {"Model":"Model B","Class":"vans","Instances":1377,"Precision":0.201,"Recall":0.191,"mAP50":0.099},

    {"Model":"Model C","Class":"jeepneys","Instances":480,"Precision":0.797,"Recall":0.710,"mAP50":0.779},
    {"Model":"Model C","Class":"bus","Instances":279,"Precision":0.732,"Recall":0.656,"mAP50":0.727},
    {"Model":"Model C","Class":"car","Instances":8965,"Precision":0.847,"Recall":0.767,"mAP50":0.837},
    {"Model":"Model C","Class":"motorcycle","Instances":4048,"Precision":0.697,"Recall":0.577,"mAP50":0.635},
    {"Model":"Model C","Class":"trucks","Instances":580,"Precision":0.755,"Recall":0.667,"mAP50":0.722},
    {"Model":"Model C","Class":"vans","Instances":1377,"Precision":0.772,"Recall":0.704,"mAP50":0.773},
])

per_class

In [ ]:
from IPython.display import Image, display

runs = {
    "Model A": r"C:\Users\Arjun\Documents\yolo_vehicle_project\outputs\model_A\confusion_matrix.png",
    "Model B": r"C:\Users\Arjun\Documents\yolo_vehicle_project\outputs\model_B\confusion_matrix.png",
    "Model C": r"C:\Users\Arjun\Documents\yolo_vehicle_project\outputs\model_C\confusion_matrix.png",
}

for name, path in runs.items():
    print(name)
    display(Image(filename=path))

In [ ]:
import pandas as pd

for name, folder in [("Model A","model_A"), ("Model B","model_B"), ("Model C","model_C")]:
    csv_path = rf"C:\Users\Arjun\Documents\yolo_vehicle_project\outputs\{folder}\results.csv"
    df = pd.read_csv(csv_path)
    print(f"--- {name}: Final Epoch ---")
    print(df.tail(1).to_string(index=False))
    print()